In [ ]:
# Load env variables and create client

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()

model = "claude-haiku-4-5"
tokens = 1000

In [ ]:
# Helper functions

def add_user_msg(messages, text):
    user_message = {"role": "user", "content" : text}
    messages.append(user_message)

def add_assit_msg(messages,text):
    assistant_message = {"role": "assistant", "content" : text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature= 1.0, stop_sequences=[]):
    params={
        "model": model,
        "max_tokens": tokens,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    if system:
        params["system"]= system
        
    response = client.messages.create(**params)
    return response.content[0].text


In [ ]:
# define a prompt to generate a evaluation dataset

import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_msg(messages, prompt)
    add_assit_msg(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.load(text)

In [ ]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent= 2)

In [ ]:
# This function takes a test case and merges it with our prompt template
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation

"""
    
    messages = []
    add_user_msg(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def model_grader(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_msg(messages,eval_prompt)
    add_assit_msg(messages, "```json")
    eval_result = chat(messages, stop_sequences="```")

    return json.loads(eval_result)

In [ ]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [ ]:
# This function orchestrates running a single test case and grading the result
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    model_grade = model_grader(test_case,output)
    code_grade_score = grade_syntax(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    score = (model_score + code_grade_score) / 2
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning" : reasoning
    }

In [ ]:
#This function coordinates the entire evaluation process

from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    avg_score = mean([results["score"] for result in results])
    print("Avrege score: ", avg_score)
    return results


In [ ]:
# To execute our evaluation pipeline, we load our dataset and run it through our functions

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results =run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))